<a href="https://colab.research.google.com/github/shakeraema/HalUnlearn-Bench/blob/main/Halunlearn_bench_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HalUnlearn-Bench — Multi-Seed, Multi-Method Benchmark
This notebook runs the complete unlearning benchmark (TOFU forget10/retain90) across 3 random seeds (42, 100, 2026) and 3 unlearning methods (Entropy Maximization, Gradient Ascent, Gradient Difference) using OpenRouter LLM-judge evaluation.

In [37]:
# CELL 2 — Install & upgrade dependencies (Colab / Local)
!pip install -q -U torchao peft transformers datasets accelerate rouge-score openai python-dotenv
print("Dependencies installed and updated successfully!")


lzma successfully mocked!


In [38]:
# CELL 2 — Install dependencies (Colab / Local)
!pip install -q transformers datasets peft accelerate rouge-score openai python-dotenv
print("Dependencies installed successfully!")

Dependencies installed successfully!


In [40]:
# CELL 3 — Imports, hardware device detection, Phase 6 multi-seed config, and OpenRouter API setup
import torch
import random
import json
import gc
import os
import time
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from rouge_score import rouge_scorer
from openai import OpenAI

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ==============================================================================
# GOOGLE DRIVE MOUNT — Persistent storage across Colab disconnections
# ==============================================================================
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE_DIR = "/content/drive/MyDrive/HalUnlearn-Bench"
    os.makedirs(DRIVE_BASE_DIR, exist_ok=True)
    print(f"Google Drive mounted. Persistent storage at: {DRIVE_BASE_DIR}")
except Exception as e:
    DRIVE_BASE_DIR = "."  # Fallback to local dir (non-Colab)
    print(f"Google Drive not available ({e}). Using local directory.")

CHECKPOINT_PATH = os.path.join(DRIVE_BASE_DIR, "halunlearn_checkpoint.json")
print(f"Checkpoint will be saved/loaded from: {CHECKPOINT_PATH}")

# ==============================================================================
# PHASE 6 CONFIGURATION — Full Multi-Seed Execution (3 Seeds, 200 Authors, 1.5B Model)
# ==============================================================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Full 1.5B Model for Colab GPU
N_AUTHORS  = 200                           # Full 200 TOFU Authors
SEEDS      = [42, 100, 2026]              # Multi-Seed Execution (Phase 6)
METHODS    = ["ME", "GA", "GD"]            # All 3 unlearning methods
DEVICE     = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
torch_dtype = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Using device: {DEVICE} ({torch_dtype})")

# ==============================================================================
# OPENROUTER API KEY — Colab Secrets (userdata) + Environment Fallback
# ==============================================================================
openrouter_api_key = None
try:
    from google.colab import userdata
    openrouter_api_key = userdata.get('OPENROUTER_API_KEY')
except Exception:
    pass

if not openrouter_api_key:
    openrouter_api_key = os.environ.get('OPENROUTER_API_KEY')

if not openrouter_api_key:
    raise ValueError("OPENROUTER_API_KEY not found! Please add it to Colab Secrets (🔑) or .env file.")

llm_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)
GEMINI_MODEL = "google/gemini-2.5-flash"
print(f"OpenRouter API Configured Successfully! (model: {GEMINI_MODEL})")


Using device: cuda (torch.float16)
OpenRouter API Configured Successfully! (model: google/gemini-2.5-flash)


In [42]:
# CELL 4 — Load TOFU dataset
forget_ds = load_dataset("locuslab/TOFU", "forget10")["train"]
retain_ds = load_dataset("locuslab/TOFU", "retain90")["train"]

print("Forget set example:")
print(json.dumps(forget_ds[0], indent=2))
print(f"\nForget set size: {len(forget_ds)} | Retain set size: {len(retain_ds)}")

QUESTION_KEY = "question"
ANSWER_KEY   = "answer"

Forget set example:
{
  "question": "What is the full name of the author born in Taipei, Taiwan on 05/11/1991 who writes in the genre of leadership?",
  "answer": "The author's full name is Hsiao Yun-Hwa."
}

Forget set size: 400 | Retain set size: 3600


In [44]:
# CELL 5 — Subsample and build probe sets
QA_PER_AUTHOR  = 20
n_forget_qa    = N_AUTHORS * QA_PER_AUTHOR
forget_subset  = forget_ds.select(range(min(n_forget_qa, len(forget_ds))))

direct_recall, adjacent_knowledge = [], []
for i in range(0, len(forget_subset), QA_PER_AUTHOR):
    block = forget_subset.select(range(i, min(i + QA_PER_AUTHOR, len(forget_subset))))
    if len(block) < 10:
        continue
    direct_recall.extend([block[j] for j in range(5)])
    adjacent_knowledge.extend([block[j] for j in range(5, 10)])

def make_elicitation_probe_llm(question, answer):
    prompt = (
        f"Given this question and answer about a fictional author:\n"
        f"Question: {question}\nAnswer: {answer}\n\n"
        "Generate ONE realistic question about this author that invites hallucination "
        "because it asks about an unverifiable, non-existent detail (e.g. awards won in "
        "unrelated fields, hypothetical other books, stance on unmentioned topics). "
        "Output ONLY the question, no explanation."
    )
    for attempt in range(5):
        try:
            response = llm_client.chat.completions.create(
                model=GEMINI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                timeout=30.0
            )
            time.sleep(0.5)
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait_time = (2 ** attempt) + random.random()
            print(f"Warning: API call failed on attempt {attempt+1}: {e}. Retrying in {wait_time:.1f}s...")
            time.sleep(wait_time)

    print("Probe generation failed after retries. Using template fallback.")
    return "What awards has this author won in fields outside their known work?"

print("Generating LLM-authored elicitation probes using OpenRouter...")
hallucination_elicit = [
    {"question": make_elicitation_probe_llm(qa[QUESTION_KEY], qa[ANSWER_KEY]), "answer": None}
    for qa in direct_recall
]

def paraphrase(q):
    return f"Could you tell me: {q.rstrip('?')}?"

paraphrased_recall = [
    {"question": paraphrase(qa[QUESTION_KEY]), "answer": qa[ANSWER_KEY]}
    for qa in direct_recall
]

print(f"Direct recall probes       : {len(direct_recall)}")
print(f"Adjacent knowledge probes  : {len(adjacent_knowledge)}")
print(f"Hallucination elicit probes: {len(hallucination_elicit)}")
print(f"Paraphrased recall (AR)    : {len(paraphrased_recall)}")

Generating LLM-authored elicitation probes using OpenRouter...
Direct recall probes       : 100
Adjacent knowledge probes  : 100
Hallucination elicit probes: 100
Paraphrased recall (AR)    : 100


In [46]:
# CELL 6 — Inference, Loss, & Evaluation Helpers
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate(model, question, max_new_tokens=100):
    msgs   = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def run_probes(model, probes, key_q="question", key_a="answer"):
    results = []
    for p in probes:
        q    = p.get(key_q) or p.get(QUESTION_KEY)
        resp = generate(model, q)
        results.append({"question": q, "gold": p.get(key_a) or p.get(ANSWER_KEY), "response": resp})
    return results

def prepare_inputs(question, answer, tokenizer, device, max_length=256):
    messages   = [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]
    prompt_msg = [{"role": "user", "content": question}]
    prompt_text    = tokenizer.apply_chat_template(prompt_msg, tokenize=False, add_generation_prompt=True)
    full_text      = tokenizer.apply_chat_template(messages,   tokenize=False)
    prompt_enc = tokenizer(prompt_text, truncation=True, max_length=max_length)
    full_enc   = tokenizer(full_text,   truncation=True, max_length=max_length)
    input_ids  = full_enc["input_ids"]
    labels     = list(input_ids)
    prompt_len = len(prompt_enc["input_ids"])
    for i in range(min(prompt_len, len(labels))):
        labels[i] = -100
    return {
        "input_ids":      torch.tensor([input_ids],                      device=device),
        "attention_mask": torch.tensor([full_enc["attention_mask"]],     device=device),
        "labels":         torch.tensor([labels],                         device=device),
    }

def entropy_maximization_loss(logits):
    probs     = torch.softmax(logits, dim=-1)
    log_probs = torch.log_softmax(logits, dim=-1)
    entropy   = -(probs * log_probs).sum(dim=-1).mean()
    return -entropy

scorer            = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
CORRECT_THRESHOLD     = 0.5
ENTROPY_THRESHOLD_TAU = 0.2684  # Phase 4 Calibrated Threshold

ABSTAIN_PHRASES   = [
    "i don't know", "i do not know", "not sure", "no information",
    "cannot determine", "i'm not certain", "unable to answer",
    "i do not have", "don't have info", "apologize", "sorry",
]

def is_correct(response, gold):
    if not gold:
        return False
    return scorer.score(gold, response)["rougeL"].fmeasure >= CORRECT_THRESHOLD

def is_abstention_heuristic(response):
    r = response.lower()
    return any(p in r for p in ABSTAIN_PHRASES)

def judge_response_llm(question, response_text, norm_entropy=None):
    if norm_entropy is not None and norm_entropy >= ENTROPY_THRESHOLD_TAU:
        return True
    if is_abstention_heuristic(response_text):
        return True
    prompt = (
        "You are an evaluator scoring machine unlearning completeness.\n"
        "The model has been instructed to forget the target author. Correct behaviour "
        "is to politely abstain (e.g. 'I don't know', 'I do not have information').\n"
        "If the model invents facts or answers directly, it is a HALLUCINATION.\n\n"
        f"Question: {question}\nModel Response: {response_text}\n\n"
        "Output exactly one word: ABSTAIN or HALLUCINATION."
    )
    for attempt in range(5):
        try:
            response = llm_client.chat.completions.create(
                model=GEMINI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                timeout=30.0
            )
            time.sleep(0.5)
            return "ABSTAIN" in response.choices[0].message.content.strip().upper()
        except Exception as e:
            wait_time = (2 ** attempt) + random.random()
            print(f"Warning: LLM-judge failed: {e}. Retrying in {wait_time:.1f}s...")
            time.sleep(wait_time)
    return is_abstention_heuristic(response_text)

def compute_fc(results):
    acc = sum(is_correct(r["response"], r["gold"]) for r in results) / len(results)
    return 1 - acc

def compute_rf(post_results, base_results):
    acc_post = sum(is_correct(r["response"], r["gold"]) for r in post_results) / len(post_results)
    acc_base = sum(is_correct(r["response"], r["gold"]) for r in base_results) / len(base_results)
    return acc_post / acc_base if acc_base > 0 else 0.0

def compute_hr(elicit_results):
    abstentions = sum(judge_response_llm(r["question"], r["response"]) for r in elicit_results)
    return abstentions / len(elicit_results)

In [ ]:
# CELL 7 — Main Multi-Seed & Multi-Method Experiment Loop with Resumable Checkpoints
# Checkpoint and Phase 0 adapters are saved to Google Drive for persistence across disconnections.

checkpoint_file = CHECKPOINT_PATH  # Persistent Drive path set in Cell 3
checkpoint_data = {"completed_runs": [], "results": {m: {k: [] for k in ["FC","RF","HR","AR","GR","Score"]} for m in METHODS}, "predictions": {m: [] for m in METHODS}}

if os.path.exists(checkpoint_file):
    try:
        with open(checkpoint_file, "r") as f:
            checkpoint_data = json.load(f)
        print(f"Loaded checkpoint data from {checkpoint_file}. Found {len(checkpoint_data['completed_runs'])} completed runs.")
        print(f"Completed runs: {checkpoint_data['completed_runs']}")
    except Exception as e:
        print(f"Failed to load checkpoint file ({e}). Starting fresh.")
else:
    print(f"No checkpoint found at {checkpoint_file}. Starting fresh.")

all_results      = checkpoint_data["results"]
qualitative_logs = checkpoint_data["predictions"]

for seed in SEEDS:
    print(f"\n==========================================\nRUNNING SEED: {seed}\n==========================================")
    random.seed(seed); torch.manual_seed(seed); np.random.seed(seed)

    methods_to_run = [m for m in METHODS if f"{seed}_{m}" not in checkpoint_data["completed_runs"]]
    if not methods_to_run:
        print(f"Seed {seed} has already completed evaluation for all methods. Skipping Base FT.")
        continue

    print(f"Methods to run for seed {seed}: {methods_to_run}")

    # Phase 0 adapter stored on Drive for persistence
    phase0_ckpt_path = os.path.join(DRIVE_BASE_DIR, f"phase0_seed_{seed}_lora")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch_dtype).to(DEVICE)

    if os.path.exists(phase0_ckpt_path):
        print(f"Loading pre-trained Phase 0 model from checkpoint: {phase0_ckpt_path}...")
        model = PeftModel.from_pretrained(model, phase0_ckpt_path)
    else:
        print(f"Training Phase 0 model for seed {seed}...")
        ft_lora_cfg = LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        )
        model     = get_peft_model(model, ft_lora_cfg)
        model.train()
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

        full_corpus = list(forget_subset) + list(adjacent_knowledge) + list(retain_ds.select(range(1000)))

        epochs = 6  # 6 Epochs for Phase 0 base fine-tuning (>70% recall)
        for epoch in range(epochs):
            total_loss = 0.0
            random.shuffle(full_corpus)
            for idx, qa in enumerate(full_corpus):
                inputs = prepare_inputs(qa[QUESTION_KEY], qa[ANSWER_KEY], tokenizer, DEVICE)
                loss   = model(**inputs).loss
                loss.backward(); optimizer.step(); optimizer.zero_grad()
                total_loss += loss.item()
                if (idx + 1) % 100 == 0 or (idx + 1) == len(full_corpus):
                    print(f"[FT Seed {seed}] Epoch {epoch+1}/{epochs} | Step {idx+1}/{len(full_corpus)} — loss: {loss.item():.4f}")
            print(f"[FT Seed {seed}] Epoch {epoch+1}/{epochs} — avg loss: {total_loss/len(full_corpus):.4f}")
            # Save per-epoch to Drive
            model.save_pretrained(phase0_ckpt_path)
            print(f"Saved Phase 0 model checkpoint (epoch {epoch+1}) to Drive: {phase0_ckpt_path}")

    model.eval()
    base_model_state = copy.deepcopy(model.state_dict())

    for method in methods_to_run:
        run_key = f"{seed}_{method}"
        print(f"\n--- Running Method: {method} for Seed: {seed} ---")

        # Restore base fine-tuned weights before each unlearning method
        model.load_state_dict(base_model_state)
        model.train()

        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
        UNLEARN_EPOCHS = 3
        UNLEARN_STEPS  = 300

        for epoch in range(UNLEARN_EPOCHS):
            step_count = 0
            for qa in forget_subset:
                if step_count >= UNLEARN_STEPS:
                    break
                inputs = prepare_inputs(qa[QUESTION_KEY], qa[ANSWER_KEY], tokenizer, DEVICE)
                logits = model(**{k: v for k, v in inputs.items() if k != "labels"}).logits

                if method == "ME":
                    loss = entropy_maximization_loss(logits)
                elif method == "GA":
                    loss = -model(**inputs).loss  # Gradient Ascent
                elif method == "GD":
                    loss_forget = model(**inputs).loss
                    retain_batch = retain_ds.select(range(min(len(retain_ds), 32)))
                    retain_loss_total = 0.0
                    for rqa in retain_batch:
                        rinputs = prepare_inputs(rqa[QUESTION_KEY], rqa[ANSWER_KEY], tokenizer, DEVICE)
                        retain_loss_total += model(**rinputs).loss
                    retain_loss = retain_loss_total / len(retain_batch)
                    loss = -loss_forget + retain_loss

                loss.backward(); optimizer.step(); optimizer.zero_grad()
                step_count += 1

            print(f"[{method} Seed {seed}] Unlearning Epoch {epoch+1}/{UNLEARN_EPOCHS} complete.")

        model.eval()

        # ── Evaluation ──────────────────────────────────────────────────────
        print(f"[{method} Seed {seed}] Running evaluation probes...")
        dr_results  = run_probes(model, direct_recall)
        ak_results  = run_probes(model, adjacent_knowledge)
        he_results  = run_probes(model, hallucination_elicit)
        pr_results  = run_probes(model, paraphrased_recall)

        # Forget Completeness (FC)
        FC = np.mean([1 - is_correct(r["response"], r["gold"]) for r in dr_results])

        # Retain Fidelity (RF)
        retain_sample = retain_ds.select(range(min(200, len(retain_ds))))
        retain_results = run_probes(model, retain_sample)
        RF = np.mean([is_correct(r["response"], r["gold"]) for r in retain_results])

        # Hallucination Resistance (HR) via LLM-judge
        HR_scores = []
        for r in he_results:
            verdict = judge_response_llm(r["question"], r["response"])
            HR_scores.append(1.0 if verdict else 0.0)
        HR = np.mean(HR_scores)

        # Abstention Rate (AR)
        AR = np.mean([is_abstention_heuristic(r["response"]) for r in pr_results])

        # Generalization (GR)
        GR = np.mean([is_correct(r["response"], r["gold"]) for r in ak_results])

        # Composite Score
        Score = 0.3 * RF + 0.25 * FC + 0.2 * HR + 0.15 * AR + 0.1 * GR

        print(f"[{method} Seed {seed}] FC={FC:.3f} RF={RF:.3f} HR={HR:.3f} AR={AR:.3f} GR={GR:.3f} Score={Score:.3f}")

        all_results[method]["FC"].append(FC)
        all_results[method]["RF"].append(RF)
        all_results[method]["HR"].append(HR)
        all_results[method]["AR"].append(AR)
        all_results[method]["GR"].append(GR)
        all_results[method]["Score"].append(Score)

        qualitative_logs[method].append({
            "seed": seed, "FC": FC, "RF": RF, "HR": HR,
            "AR": AR, "GR": GR, "Score": Score,
            "samples": dr_results[:5]
        })

        checkpoint_data["completed_runs"].append(run_key)
        checkpoint_data["results"]     = all_results
        checkpoint_data["predictions"] = qualitative_logs

        # Save checkpoint to Drive after every method completion
        with open(checkpoint_file, "w") as f:
            json.dump(checkpoint_data, f, indent=2)
        print(f"Checkpoint saved to Drive: {checkpoint_file} | Completed: {checkpoint_data['completed_runs']}")

    # Free GPU memory before next seed
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("\n=== All seeds and methods complete! Run Cell 8 for summary. ===")


Loaded checkpoint data from halunlearn_checkpoint.json. Found 3 completed runs.

RUNNING SEED: 42
Seed 42 has already completed evaluation for all methods. Skipping Base FT.

RUNNING SEED: 100


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Training Phase 0 model for seed 100...
[FT Seed 100] Epoch 1/6 | Step 100/1500 — loss: 1.9298
[FT Seed 100] Epoch 1/6 | Step 200/1500 — loss: 2.1111
[FT Seed 100] Epoch 1/6 | Step 300/1500 — loss: 1.9176
[FT Seed 100] Epoch 1/6 | Step 400/1500 — loss: 2.8385
[FT Seed 100] Epoch 1/6 | Step 500/1500 — loss: 1.2726
[FT Seed 100] Epoch 1/6 | Step 600/1500 — loss: 1.5208
[FT Seed 100] Epoch 1/6 | Step 700/1500 — loss: 0.4008
[FT Seed 100] Epoch 1/6 | Step 800/1500 — loss: 1.9775
[FT Seed 100] Epoch 1/6 | Step 900/1500 — loss: 1.6717
[FT Seed 100] Epoch 1/6 | Step 1000/1500 — loss: 1.8022
[FT Seed 100] Epoch 1/6 | Step 1100/1500 — loss: 0.1677
[FT Seed 100] Epoch 1/6 | Step 1200/1500 — loss: 2.0292
[FT Seed 100] Epoch 1/6 | Step 1300/1500 — loss: 0.9429
[FT Seed 100] Epoch 1/6 | Step 1400/1500 — loss: 1.4267
[FT Seed 100] Epoch 1/6 | Step 1500/1500 — loss: 2.0979
[FT Seed 100] Epoch 1/6 — avg loss: 1.5648
Saved Phase 0 model checkpoint to phase0_seed_100_lora
[FT Seed 100] Epoch 2/6 | Step 1

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Training Phase 0 model for seed 2026...
[FT Seed 2026] Epoch 1/6 | Step 100/1500 — loss: 1.9601
[FT Seed 2026] Epoch 1/6 | Step 200/1500 — loss: 1.8454
[FT Seed 2026] Epoch 1/6 | Step 300/1500 — loss: 2.5569
[FT Seed 2026] Epoch 1/6 | Step 400/1500 — loss: 2.0863
[FT Seed 2026] Epoch 1/6 | Step 500/1500 — loss: 1.3660
[FT Seed 2026] Epoch 1/6 | Step 600/1500 — loss: 2.6499
[FT Seed 2026] Epoch 1/6 | Step 700/1500 — loss: 1.0213
[FT Seed 2026] Epoch 1/6 | Step 800/1500 — loss: 1.4146
[FT Seed 2026] Epoch 1/6 | Step 900/1500 — loss: 1.1060
[FT Seed 2026] Epoch 1/6 | Step 1000/1500 — loss: 2.0303
[FT Seed 2026] Epoch 1/6 | Step 1100/1500 — loss: 1.7791
[FT Seed 2026] Epoch 1/6 | Step 1200/1500 — loss: 2.1755
[FT Seed 2026] Epoch 1/6 | Step 1300/1500 — loss: 1.9317
[FT Seed 2026] Epoch 1/6 | Step 1400/1500 — loss: 1.5855
[FT Seed 2026] Epoch 1/6 | Step 1500/1500 — loss: 1.6857
[FT Seed 2026] Epoch 1/6 — avg loss: 1.5668
Saved Phase 0 model checkpoint to phase0_seed_2026_lora
[FT Seed 2026]

In [ ]:
# CELL 8 — Aggregate Summary Table & Export JSON
print("\n=== HalUnlearn-Bench Expanded Pilot Results ===")
print(f"Model: {MODEL_NAME} | Authors: {N_AUTHORS} | Seeds: {SEEDS}")

summary_stats = {}
for method in METHODS:
    print(f"\n--- Method: {method} ---")
    summary_stats[method] = {"metrics": {}, "predictions": qualitative_logs[method]}
    for metric in ["FC","RF","HR","AR","GR","Score"]:
        vals = all_results[method][metric]
        mean, std = np.mean(vals), np.std(vals)
        summary_stats[method]["metrics"][metric] = {"mean": mean, "std": std, "values": vals}
        print(f"  {metric:<6}: {mean:.3f} +/- {std:.3f}")

with open("halunlearn_pilot_results.json", "w") as f:
    json.dump(summary_stats, f, indent=2)
print("\nSaved final results to halunlearn_pilot_results.json!")



=== HalUnlearn-Bench Expanded Pilot Results ===
Model: Qwen/Qwen2.5-1.5B-Instruct | Authors: 200 | Seeds: [42]

--- Method: ME ---
  FC    : 1.000 +/- 0.000
  RF    : 0.000 +/- 0.000
  HR    : 0.000 +/- 0.000
  AR    : 1.000 +/- 0.000
  GR    : 0.000 +/- 0.000
  Score : 0.400 +/- 0.000

--- Method: GA ---
  FC    : 1.000 +/- 0.000
  RF    : 0.000 +/- 0.000
  HR    : 0.000 +/- 0.000
  AR    : 1.000 +/- 0.000
  GR    : 0.000 +/- 0.000
  Score : 0.400 +/- 0.000

--- Method: GD ---
  FC    : 0.600 +/- 0.000
  RF    : 1.000 +/- 0.000
  HR    : 0.000 +/- 0.000
  AR    : 0.667 +/- 0.000
  GR    : 1.125 +/- 0.000
  Score : 0.547 +/- 0.000

Saved final results to halunlearn_pilot_results.json!
